#### Final Project Option 2:
##### Using text embeddings for matching the ingredient names

#### Part 1: Setup

##### Create the BQ datasets


In [ ]:
from google.cloud import bigquery

project_id = "segfault-434120"
dataset = "fin_skincare_products"
region = "us-central1"

bq_client = bigquery.Client()

dataset_id = bigquery.Dataset(f"{project_id}.{dataset}")
dataset_id.location = region
resp = bq_client.create_dataset(dataset_id, exists_ok=True)
print("Created dataset {}.{}".format(bq_client.project, resp.dataset_id))

Created dataset segfault-434120.fin_skincare_products


In [ ]:
from google.cloud import bigquery

project_id = "segfault-434120"
dataset = "ai_models"
region = "us-central1"

bq_client = bigquery.Client()

dataset_id = bigquery.Dataset(f"{project_id}.{dataset}")
dataset_id.location = region
resp = bq_client.create_dataset(dataset_id, exists_ok=True)
print("Created dataset {}.{}".format(bq_client.project, resp.dataset_id))

Created dataset segfault-434120.ai_models


##### Register the embeddings model with BQ
##### Before running the next cell, create a remote connection to Vertex AI and then grant the service account associated with the connection the "Vertex AI User" role

In [ ]:
%%bigquery
create or replace model ai_models.text_embedding
remote with connection `projects/segfault-434120/locations/us-central1/connections/embedding_connection_id`
options (endpoint = 'text-embedding-004');

Query is running:   0%|          |

""


#### Part 2: Review the input data

##### The objective is to match up the cosmetic_ingredient such that any two ingredient names in disease treatments that correspond to the same cosmetic ingredient name get assigned a common key. For example, 'Aloe' and 'Aloe Extract' are the same entity, so they would be assigned the same key and similarly for 'Water' and 'Mineral Water'.

In [ ]:
%%bigquery
select ingredient_unique_name from skincare_products_stg.cosmetic_ingredient order by ingredient_unique_name

Query is running:   0%|          |

Downloading:   0%|          |

,ingredient_unique_name
0,DISODIUM TETRAMETHYLHEXADECENYLCYSTEINE FORM...
1,(LIQUIDAMBAR STYRACIFLUA/TRIBULUS TERRESTRIS)...
2,ACRYLATES/VA/VINYL NEODECANOATE COPOLYMER
3,ASTROCARYUM VULGARE SEED BUTTER
4,BARLEY SH-POLYPEPTIDE-17
...,...
28707,ACRYLOYL DIMETHYL TAURATE/MELAMINE/PEG-6 METH...
28708,BACILLUS/(ASPARAGUS COCHINCHINENSIS/LYCIUM CH...
28709,LACTOBACILLUS/HYDROLYZED [HONEY/SUGAR CANE/(A...
28710,LACTOBACILLUS/RHODOPSEUDOMONAS/SACCHAROMYCES/...


In [ ]:
%%bigquery
select distinct regexp_replace(upper(treatment), r'["]', '') as treatment
from `skincare_products_stg.dermatologic_disease`, unnest(treatments) as treatment
order by treatment

Query is running:   0%|          |

Downloading:   0%|          |

,treatment
0,5-FLUOROURACIL
1,ABROCITINIB
2,ACCOLATE
3,ACCUTANE
4,ACETAMINOPHEN
...,...
236,VITAMIN D
237,WHEAT GERM OIL
238,ZINC OXIDE
239,ZINC PYRITHIONE


#### Part 3: Create the embeddings

#### Create the embeddings on the cosmetic_ingredient ingredient_unique_name.
###### More details on the `ml.generate_embedding()`: https://cloud.google.com/bigquery/docs/reference/standard-sql/bigqueryml-syntax-generate-embedding#text-embedding

In [ ]:
%%bigquery
create or replace table fin_skincare_products.tmp_cosmetic_ingredient_embedding as (
select
  ingredient_unique_name,
  ml_generate_embedding_result as embedding
from
  ml.generate_embedding(
    model ai_models.text_embedding,
    (select ingredient_unique_name, ingredient_unique_name as content from skincare_products_stg.cosmetic_ingredient
    where ingredient_unique_name is not null),
    struct('CLUSTERING' as task_type)
  )
);

Query is running:   0%|          |

""


In [ ]:
%%bigquery
create or replace table fin_skincare_products.tmp_dermatologic_treatments_embedding as (
select
  treatment,
  ml_generate_embedding_result as embedding
from
  ml.generate_embedding(
    model ai_models.text_embedding,
    (select distinct regexp_replace(upper(treatment), r'["]', '') as treatment, treatment as content
    from `skincare_products_stg.dermatologic_disease`, unnest(treatments) as treatment
    order by treatment),
    struct('clustering' as task_type)
  )
);

Query is running:   0%|          |

""


In [ ]:
%%bigquery
select * from fin_skincare_products.tmp_dermatologic_treatments_embedding

Query is running:   0%|          |

Downloading:   0%|          |

,treatment,embedding
0,BEXAROTENE,"[-0.019608618691563606, 0.001017242087982595, ..."
1,CIPROFLOXACIN,"[-0.0008821728406473994, 0.020312802866101265,..."
2,RAPAMYCIN,"[-0.036502957344055176, -0.011952044442296028,..."
3,UPADACITINIB,"[-0.061737824231386185, -0.0012458201963454485..."
4,ADALIMUMAB,"[0.010700120590627193, -0.049337029457092285, ..."
...,...,...
236,PROPYLENE GLYCOL,"[-0.0024794472847133875, 0.009664020501077175,..."
237,TRIAMCINOLONE,"[0.025986840948462486, -0.029033180326223373, ..."
238,BEE'S WAX,"[0.020113429054617882, 0.03043048270046711, 0...."
239,PENTOXIFYLLINE,"[-0.02443297766149044, 0.05361180007457733, -0..."


##### Note: the embedding gets created on the `content` field. Having a `content` field is required when calling `ml.generate_embedding()`

In [ ]:
%%bigquery
select * from fin_skincare_products.tmp_cosmetic_ingredient_embedding

Query is running:   0%|          |

Downloading:   0%|          |

,ingredient_unique_name,embedding
0,"CIS-3,5,5-TRIMETHYLCYCLOHEXANOL","[-0.0177154541015625, 0.07035285979509354, -0...."
1,CAPRYLOYL TETRAPEPTIDE-60 NORLEUCYL TYROSYL D-...,"[0.01593017578125, 0.05852653831243515, -0.026..."
2,ETHYLBISIMINOMETHYLGUAIACOL MANGANESE CHLORIDE,"[-0.02679443359375, -0.0024053805973380804, -0..."
3,BRASSICA JUNCEA SPROUT EXTRACT,"[-0.015363693237304688, -0.013038398697972298,..."
4,ISOBUTYLPARABEN,"[-0.04582977294921875, 0.0367048978805542, -0...."
...,...,...
28707,FUCUS SERRATUS EXTRACT,"[-0.003233432536944747, 0.0022504765074700117,..."
28708,LINUM USITATISSIMUM HULL EXTRACT,"[-0.023990629240870476, -0.009610319510102272,..."
28709,EUCALYPTUS SPECIES LEAF OIL,"[-0.017246244475245476, -0.022974276915192604,..."
28710,OLETH-11,"[-0.005318641196936369, -0.0008330405107699335..."


#### Part 4: Match similar businesses
###### More details on the `vector_search()` function: https://cloud.google.com/bigquery/docs/reference/standard-sql/search_functions#**vector_search**

In [ ]:
%%bigquery
create or replace table fin_skincare_products.tmp_treatment_ingredient_nearest_neighbor as
select query.treatment as treatment_name, base.ingredient_unique_name as nearest_neighbor, distance
from
  vector_search(
    table fin_skincare_products.tmp_cosmetic_ingredient_embedding,
    'embedding',
    table fin_skincare_products.tmp_dermatologic_treatments_embedding,
    'embedding',
    top_k => 2,  # might switch to 1
    distance_type => 'COSINE')
where query.treatment != base.ingredient_unique_name
order by distance

Query is running:   0%|          |

""


In [ ]:
%%bigquery
select * from fin_skincare_products.tmp_treatment_ingredient_nearest_neighbor
order by distance

Query is running:   0%|          |

Downloading:   0%|          |

,treatment_name,nearest_neighbor,distance
0,BEE'S WAX,BEESWAX,0.085300
1,CAPASAICIN,CAPSAICIN,0.113600
2,DIOSMIN,DIOSMETIN,0.133921
3,CENTELLA ASIATICA,CENTELLA ASIATICA EXTRACT,0.136407
4,BENZYL ALCOHOL,BEHENYL ALCOHOL,0.140299
...,...,...,...
429,ACCOLATE,TALLOW,0.491453
430,ANTIVIRAL MEDICATIONS,TRIFLUOROMETHYLPHENETHYL MESALAZINE,0.495441
431,ANTIVIRAL MEDICATIONS,CYCLOVIROBUXINE,0.514846
432,INTRALESIONAL INJECTIONS,NEURAL EXTRACT,0.521325


#### Part 5: Assign a common key to each cluster of nearest neighbors

##### For example, map 'BEE'S WAX' and 'BEESWAX' to the same key

##### But first, look at the data to see what is the maximum distance we should accept to consider two business names as the same entity

In [ ]:
%%bigquery
select * from fin_skincare_products.tmp_treatment_ingredient_nearest_neighbor
where distance between 0.1 and 0.2
order by distance desc

Query is running:   0%|          |

Downloading:   0%|          |

,treatment_name,nearest_neighbor,distance
0,RETINOIDS,RETINOIC ACID,0.198282
1,POTASSIUM IODIDE,SODIUM IODIDE,0.195354
2,METHYLAMINOLEVULINATE,METHYL AMINOLEVULINATE HCL,0.193768
3,TRICHLOROACETIC ACID,TRIFLUOROACETIC ACID,0.192017
4,ZINC OXIDE,ZINC HYDROXIDE,0.191729
5,PRAMOXINE,PRAMOXINE HCL,0.191502
6,N-ACETYLCYSTEINE,ACETYL CYSTEINE,0.189478
7,DIPHENHYDRAMINE,DIPHENHYDRAMINE HCL,0.189234
8,PROPYLENE GLYCOL,PROPYLENE GLYCOL PROPYL ETHER,0.180093
9,QUINACRINE,QUININE,0.176915


##### It appears that 0.176 would make a decent cutoff, so we'll consider all names whose distance is < 0.176 to its nearest neighbor to be same business

In [ ]:
import pandas
import pandas_gbq
from google.cloud import bigquery

project_id = "segfault-434120"
region = "us-central1"
output_table = "fin_skincare_products.tmp_ingredient_replacements"

base_query = """select treatment_name, nearest_neighbor
from fin_skincare_products.tmp_treatment_ingredient_nearest_neighbor
where distance < 0.176 order by treatment_name
"""

bq_client = bigquery.Client()
rows = bq_client.query_and_wait(base_query)
records = []
for row in rows:
    treatment = row["treatment_name"]
    nearest_neighbor = row["nearest_neighbor"]
    records.append((treatment, nearest_neighbor))


df = pandas.DataFrame.from_records(records, columns=['treatment_name', 'nearest_neighbor'])
print(df)

pandas_gbq.to_gbq(df, output_table, project_id=project_id, if_exists="replace")

               treatment_name  \
0           ALUMINUM CHLORIDE   
1                   BEE'S WAX   
2              BENZYL ALCOHOL   
3                  CAPASAICIN   
4           CENTELLA ASIATICA   
5           CENTELLA ASIATICA   
6                     DIOSMIN   
7                     DIOSMIN   
8                     EGG OIL   
9                     RETINOL   
10                SHEA BUTTER   
11  SODIUM TETRADECYL SULFATE   
12                        TAR   
13                 TOCOPHEROL   
14             WHEAT GERM OIL   
15               ZINC SULFATE   

                                     nearest_neighbor  
0                              ALUMINUM CHLOROHYDRATE  
1                                             BEESWAX  
2                                     BEHENYL ALCOHOL  
3                                           CAPSAICIN  
4                           CENTELLA ASIATICA EXTRACT  
5                      CENTELLA ASIATICA LEAF EXTRACT  
6                                           DI

100%|██████████| 1/1 [00:00<00:00, 7281.78it/s]


In [ ]:
%%bigquery
select * from fin_skincare_products.tmp_ingredient_replacements order by treatment_name

Query is running:   0%|          |

Downloading:   0%|          |

,treatment_name,nearest_neighbor
0,ALUMINUM CHLORIDE,ALUMINUM CHLOROHYDRATE
1,BEE'S WAX,BEESWAX
2,BENZYL ALCOHOL,BEHENYL ALCOHOL
3,CAPASAICIN,CAPSAICIN
4,CENTELLA ASIATICA,CENTELLA ASIATICA EXTRACT
5,CENTELLA ASIATICA,CENTELLA ASIATICA LEAF EXTRACT
6,DIOSMIN,DIOSMETIN
7,DIOSMIN,DIOSMINE
8,EGG OIL,HYDROGENATED EGG OIL
9,RETINOL,RETINYL RETINOATE


##### Prepare the final output table
##### Note that this table would not be the final table in the intermediate layer. It is still just a temp table because there is more work that would happen after this stage to decompose it into a `Cosmetic_Ingredient` table (to represent the set of unique ingredients) and an `Disease_Ingredient_Treatment` table to represent the many-to-many relationship between `Dermatologic_Disease` and `Cosmetic_Ingredient`.

In [ ]:
%%bigquery
CREATE OR REPLACE TABLE fin_skincare_products.tmp_dermatologic_disease_cleaned AS
SELECT
  * except(treatments),
  ARRAY(
    SELECT DISTINCT REGEXP_REPLACE(UPPER(treatment), r'["]', '') -- Clean treatment names
    FROM UNNEST(treatments) AS treatment
  ) AS treatments, -- Cleaned treatments array
FROM
  skincare_products_stg.dermatologic_disease;

Query is running:   0%|          |

""


##### Replace `treatments` with `nearest_neighbor` names

In [ ]:
from google.cloud import bigquery
import pandas as pd

# Initialize the BigQuery client
client = bigquery.Client()

# Query to fetch the data from tmp_dermatologic_disease_cleaned
cleaned_query = """
SELECT *
FROM fin_skincare_products.tmp_dermatologic_disease_cleaned
"""
cleaned_df = client.query(cleaned_query).to_dataframe()

# Query to fetch the ingredient replacements
replacements_query = """
SELECT treatment_name, nearest_neighbor
FROM fin_skincare_products.tmp_ingredient_replacements
"""
replacements_df = client.query(replacements_query).to_dataframe()

# Convert the replacements into a dictionary for faster lookup
replacement_dict = dict(zip(replacements_df['treatment_name'], replacements_df['nearest_neighbor']))

# Function to apply replacements to the treatments array
def replace_treatments(treatments):
    global replacements, replacement_count
    if treatments is None:
        return None
    updated_treatments = []
    for t in treatments:
        replaced = replacement_dict.get(t, t)  # Apply replacement logic
        if replaced != t:
            replacement_count += 1  # Increment counter if a replacement was made
            replacements.add(replaced)
        updated_treatments.append(replaced)
    return updated_treatments

# Apply the replacement function to the treatments column
replacements = set()
replacement_count = 0
cleaned_df['treatments'] = cleaned_df['treatments'].apply(replace_treatments)
print(cleaned_df)

# Write the updated dataframe back to BigQuery
table_id = "fin_skincare_products.tmp_dermatologic_disease_replaced"

# Define BigQuery table schema (optional)
job_config = bigquery.LoadJobConfig(write_disposition="WRITE_TRUNCATE")

# Write the dataframe to BigQuery
job = client.load_table_from_dataframe(cleaned_df, table_id, job_config=job_config)
job.result()  # Wait for the job to complete
print(f"replacement_count: {replacement_count}")
print(replacements)
print(f"Updated table {table_id} successfully!")

                               name  \
0                          Angiomas   
1               Jiggers (Tungiasis)   
2                 Alopecia Mucinosa   
3    Congenital Adrenal Hyperplasia   
4                         Xanthomas   
..                              ...   
365                  Chemical Peels   
366               Actinic Keratosis   
367            Hot Tub Folliculitis   
368              Pyogenic Granuloma   
369             Granuloma Inguinale   

                                  physical_description  \
0    Benign growths consisting of small blood vesse...   
1    Jiggers, also known as tungiasis, are caused b...   
2    An abnormal amount of mucin accumulates in the...   
3    A disorder that affects the adrenal glands, re...   
4    Xanthomas are skin blemishes formed by fat bui...   
..                                                 ...   
365  Chemical peels are a procedure that uses vario...   
366                    Dry, scaly patches on the skin.   
367  Itch

#### Part 6: Evaluation

In [ ]:
%%bigquery
SELECT
  original_row.name,
  original_row.treatments AS original_treatments,
  replaced_row.treatments AS replaced_treatments,
  ARRAY_LENGTH(original_row.treatments) AS original_count,
  ARRAY_LENGTH(
    ARRAY(
      SELECT t FROM UNNEST(replaced_row.treatments) AS t
      EXCEPT DISTINCT
      SELECT t FROM UNNEST(original_row.treatments) AS t
    )
  ) AS replaced_count, -- Count of newly added treatments
  ARRAY(
    SELECT t FROM UNNEST(original_row.treatments) AS t
    EXCEPT DISTINCT
    SELECT t FROM UNNEST(replaced_row.treatments) AS t
  ) AS replaced_from_original,
  ARRAY(
    SELECT t FROM UNNEST(replaced_row.treatments) AS t
    EXCEPT DISTINCT
    SELECT t FROM UNNEST(original_row.treatments) AS t
  ) AS newly_added_treatments
FROM
  `fin_skincare_products.tmp_dermatologic_disease_cleaned` AS original_row
JOIN
  `fin_skincare_products.tmp_dermatologic_disease_replaced` AS replaced_row
USING (name, physical_description, causes, _data_source, _load_time)
ORDER BY replaced_count DESC;


Query is running:   0%|          |

Downloading:   0%|          |

,name,original_treatments,replaced_treatments,original_count,replaced_count,replaced_from_original,newly_added_treatments
0,Striae,"[COCOA BUTTER, SHEA BUTTER, TOCOPHEROL, CENTEL...","[COCOA BUTTER, SHEA BUTTER GLYCERIDE, TOCOPHER...",14,5,"[SHEA BUTTER, TOCOPHEROL, CENTELLA ASIATICA, E...","[SHEA BUTTER GLYCERIDE, TOCOPHERYL PHOSPHATE, ..."
1,Lipodermatosclerosis,"[DANAZOL, DIOSMIN, OXANDROLONE, PENTOXIFYLLINE]","[DANAZOL, DIOSMINE, OXANDROLONE, PENTOXIFYLLINE]",4,1,[DIOSMIN],[DIOSMINE]
2,Measles,[RETINOL],[RETINYL RETINOATE],1,1,[RETINOL],[RETINYL RETINOATE]
3,Prurigo Nodularis,"[PSORALEN, CAPASAICIN]","[PSORALEN, CAPSAICIN]",2,1,[CAPASAICIN],[CAPSAICIN]
4,Postherpetic Neuralgia,"[LIDOCAINE, CAPASAICIN, DULOXETINE, VENLAFAXIN...","[LIDOCAINE, CAPSAICIN, DULOXETINE, VENLAFAXINE...",6,1,[CAPASAICIN],[CAPSAICIN]
...,...,...,...,...,...,...,...
358,Systemic Lupus Erythematosus,"[HYDROXYCHLOROQUINE, CORTICOSTEROIDS, IMMUNOSU...","[HYDROXYCHLOROQUINE, CORTICOSTEROIDS, IMMUNOSU...",4,0,[],[]
359,Actinic Keratosis,"[AMINOLEVULINIC ACID, METHYLAMINOLEVULINATE]","[AMINOLEVULINIC ACID, METHYLAMINOLEVULINATE]",2,0,[],[]
360,Hot Tub Folliculitis,[SILVER SULFADIAZINE],[SILVER SULFADIAZINE],1,0,[],[]
361,Pyogenic Granuloma,"[TRICHLOROACETIC ACID, PODOPHYLLIN, PHENOL, SI...","[TRICHLOROACETIC ACID, PODOPHYLLIN, PHENOL, SI...",4,0,[],[]


In [ ]:
%%bigquery
WITH evaluation_table as (
  SELECT
    original_row.name,
    original_row.treatments AS original_treatments,
    replaced_row.treatments AS replaced_treatments,
    ARRAY_LENGTH(original_row.treatments) AS original_count,
    ARRAY_LENGTH(
      ARRAY(
        SELECT t FROM UNNEST(replaced_row.treatments) AS t
        EXCEPT DISTINCT
        SELECT t FROM UNNEST(original_row.treatments) AS t
      )
    ) AS replaced_count, -- Count of newly added treatments
    ARRAY(
      SELECT t FROM UNNEST(original_row.treatments) AS t
      EXCEPT DISTINCT
      SELECT t FROM UNNEST(replaced_row.treatments) AS t
    ) AS replaced_from_original,
    ARRAY(
      SELECT t FROM UNNEST(replaced_row.treatments) AS t
      EXCEPT DISTINCT
      SELECT t FROM UNNEST(original_row.treatments) AS t
    ) AS newly_added_treatments
  FROM
    `fin_skincare_products.tmp_dermatologic_disease_cleaned` AS original_row
  JOIN
    `fin_skincare_products.tmp_dermatologic_disease_replaced` AS replaced_row
  USING (name, physical_description, causes, _data_source, _load_time)
  ORDER BY replaced_count DESC
)

SELECT
(SELECT COUNT(*) FROM evaluation_table WHERE replaced_count > 0) as replacements_made,
(SELECT COUNT(*) FROM evaluation_table WHERE replaced_count = 0 and original_count != 0) as no_replacements,
(SELECT COUNT(*) FROM evaluation_table WHERE original_count > 0) as total_possible_replacements_in_disease


Query is running:   0%|          |

Downloading:   0%|          |

,replacements_made,no_replacements,total_possible_replacements_in_disease
0,17,212,229


#### Part 7: Performance comparison

##### With the prompting approach, how many matching ingredient replacements did we find and how many ingredients were left over?


In [ ]:
%%bigquery
WITH
Dermatologic_Disease as (
  select distinct regexp_replace(upper(treatment), r'["]', '') as treatment
  from `skincare_products_stg.dermatologic_disease`, unnest(treatments) as treatment
  order by treatment
),
flattened_ingredients AS (
    SELECT
        TRIM(ingredient) AS ingredient
    FROM Dermatologic_Disease,
         UNNEST(SPLIT(treatment, ',')) AS ingredient
),
total_ingredients_cte AS (
    SELECT COUNT(ingredient) AS total_ingredients
    FROM flattened_ingredients
),

replacements AS (
  SELECT
    COUNT(DISTINCT CASE
                     WHEN old_ingredient != new_ingredient THEN old_ingredient
                   END) AS replacements
  FROM
    dbt_skincare_products_int.tmp_ingredient_replacements
),

final_stats AS (
    SELECT
        r.replacements as prompted_replacements,
        t.total_ingredients
    FROM replacements r, total_ingredients_cte t
)
SELECT *
FROM final_stats;

Query is running:   0%|          |

Downloading:   0%|          |

,prompted_replacements,total_ingredients
0,209,241


In [ ]:
%%bigquery
WITH

evaluation_table as (
  SELECT
    original_row.name,
    original_row.treatments AS original_treatments,
    replaced_row.treatments AS replaced_treatments,
    ARRAY_LENGTH(original_row.treatments) AS original_count,
    ARRAY_LENGTH(
      ARRAY(
        SELECT t FROM UNNEST(replaced_row.treatments) AS t
        EXCEPT DISTINCT
        SELECT t FROM UNNEST(original_row.treatments) AS t
      )
    ) AS replaced_count, -- Count of newly added treatments
    ARRAY(
      SELECT t FROM UNNEST(original_row.treatments) AS t
      EXCEPT DISTINCT
      SELECT t FROM UNNEST(replaced_row.treatments) AS t
    ) AS replaced_from_original,
    ARRAY(
      SELECT t FROM UNNEST(replaced_row.treatments) AS t
      EXCEPT DISTINCT
      SELECT t FROM UNNEST(original_row.treatments) AS t
    ) AS newly_added_treatments
  FROM
    `fin_skincare_products.tmp_dermatologic_disease_cleaned` AS original_row
  JOIN
    `fin_skincare_products.tmp_dermatologic_disease_replaced` AS replaced_row
  USING (name, physical_description, causes, _data_source, _load_time)
  ORDER BY replaced_count DESC
),

Dermatologic_Disease AS (
  SELECT DISTINCT REGEXP_REPLACE(UPPER(treatment), r'["]', '') AS treatment
  FROM `skincare_products_stg.dermatologic_disease`, UNNEST(treatments) AS treatment
  ORDER BY treatment
),
flattened_ingredients AS (
    SELECT
        TRIM(ingredient) AS ingredient
    FROM Dermatologic_Disease,
         UNNEST(SPLIT(treatment, ',')) AS ingredient
),
total_ingredients_cte AS (
    SELECT COUNT(ingredient) AS total_ingredients
    FROM flattened_ingredients
),
replacements AS (
    SELECT COUNT(DISTINCT treatment) AS unique_treatment_replacements
    FROM (
      SELECT treatment
      FROM evaluation_table, UNNEST(newly_added_treatments) AS treatment
    )
),
final_stats AS (
    SELECT
        replacements.unique_treatment_replacements AS embeddings_replacements,
        total_ingredients_cte.total_ingredients
    FROM replacements, total_ingredients_cte
)
SELECT *
FROM final_stats;


Query is running:   0%|          |

Downloading:   0%|          |

,embeddings_replacements,total_ingredients
0,14,241


##### **Conclusion**: The embeddings approach resulted in 14 replacements, a smaller count compared to the 209 replacements generated by the prompting approach from the same set of 241 ingredient names. This highlights the potential for embeddings to produce more precise treatment ingredient name alignments. Additionally, the embeddings method demonstrated substantial efficiency gains, completing the task in 4 minutes compared to the 10+ hours required for the prompting approach.

##### Moving forward, combining both methods could be a promising avenue for further refinement. Embeddings can rapidly reduce the scope of ingredient names for matching, while prompting may enhance alignment accuracy through nuanced definition based matching. Careful integration of these methods could provide a balanced and effective solution.